In [ ]:
"""
VisualStep — QLoRA Fine-Tuning of Phi-4 Mini (via Unsloth)
===========================================================
Fine-tunes microsoft/Phi-4-mini-instruct sur le dataset VisualStep-2K
en utilisant Unsloth (compatible Python 3.13 + Colab T4).

ORDRE D'EXÉCUTION :
  Cellule 0  → Variables d'environnement (avant tout import)
  Cellule 1  → Installation des dépendances
  Cellule 2  → Imports et configuration
  Cellule 3  → Chargement et formatage du dataset
  Cellule 4  → Chargement du modèle + LoRA via Unsloth
  Cellule 5  → Entraînement
  Cellule 6  → Sauvegarde de l'adaptateur
  Cellule 7  → Inférence sur le jeu de test
  Cellule 8  → Métriques de synthèse

  CELL 1: Install dependencies ────
  CELL 0 — Environnement Variables   ──
  CELL 2: Imports and config ────
  CELL 3: Load and format dataset ──
  CELL 4: Load tokenizer ──
  CELL 5a - Training config
  CELL 5b — Training
  CELL 6 - Saving LoRa adapters
  CELL 7a: Loading the model for inference
  CELL 7b & 8 — Inference on Test Set and Metrics (statistics)
  CELL 9: BLEU and ROUGE metrics ────
  CELL 10 -  pushing model into hugging face
  CELL 11 - Constraint compliance check on the three confirmed baselines
  CELL 12  - Qualitative Comparison — VisualStep vs. Baselines

REQUIRES :
  /content/drive/MyDrive/VisualStep/adhd_dataset_final.csv

OUTPUT :
  /content/drive/MyDrive/VisualStep/model/
  /content/drive/MyDrive/VisualStep/results/test_outputs_visualstep.csv
"""

# ── CELL 1: Install dependencies ────



In [ ]:
%%capture
!pip install unsloth
!pip install datasets
!pip install "trl==0.15.2" -q
print("Done!")

# ── CELL 0 — Environnement Variables   ──
run cell 0
then run cell 1
then restart session
then run cell 0 then 2 and so on

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 0 — Variables d'environnement (obligatoire EN PREMIER)
# ══════════════════════════════════════════════════════════════════════════════
# Doit être exécutée AVANT tout autre import pour éviter l'erreur
# "rope_scaling short_factor" et les crashs torch.dynamo sur Phi-4 Mini.

# Unsloth DOIT être importé avant tout le reste
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import torch
torch._dynamo.config.disable = True
print("✅ GPU:", torch.cuda.get_device_name(0))

# ── CELL 2: Imports and config ────

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 2 — Imports et configuration
# ══════════════════════════════════════════════════════════════════════════════

import json, csv
import pandas as pd
from google.colab import drive
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastModel

drive.mount('/content/drive')

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_ID = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
DATASET_PATH = "/content/drive/MyDrive/VisualStep/adhd_dataset_final.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/VisualStep/model"
RESULTS_DIR  = "/content/drive/MyDrive/VisualStep/results"
MAX_SEQ_LEN  = 512

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SYSTEM_PROMPT = """You are an expert in ADHD classroom accommodations for primary school children aged 5-10.
Transform the teacher's spoken classroom instruction into an ADHD-adapted visual step card.

Rules:
- Maximum 5 steps
- Each step: one concrete physical action only
- Each action: maximum 6 words
- Start each step with an imperative verb
- Include a pictogram keyword in the icon field
- Output JSON only, no other text

Output format:
{"steps":[{"id":1,"action":"Open your book","detail":"page 23","icon":"book","check":true}],"n_steps":1,"support_level":"medium"}

support_level: "light" (1-2 steps), "medium" (3-4 steps), "high" (5 steps)"""

print("✅ Configuration prête")
print(f"   Modèle   : {MODEL_ID}")
print(f"   Dataset  : {DATASET_PATH}")
print(f"   Sortie   : {OUTPUT_DIR}")


# ── CELL 3: Load and format dataset ───

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 3 — Chargement et formatage du dataset
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Dataset
df = pd.read_csv(DATASET_PATH)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df)
n_train = int(n * 0.8)
n_dev   = int(n * 0.1)
df_train = df.iloc[:n_train]
df_dev   = df.iloc[n_train:n_train+n_dev]
df_test  = df.iloc[n_train+n_dev:]
print(f"Train: {len(df_train)} | Dev: {len(df_dev)} | Test: {len(df_test)}")

def format_example(row):
    target = str(row.get('output_json', row.get('output_steps', '')))
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Year: {row['year']} | Subject: {row['subject']} | Instruction: {row['input']}"},
        {"role": "assistant", "content": target},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_data = Dataset.from_list([format_example(row) for _, row in df_train.iterrows()])
dev_data   = Dataset.from_list([format_example(row) for _, row in df_dev.iterrows()])
test_data  = Dataset.from_list([format_example(row) for _, row in df_test.iterrows()])

print("✅ Dataset prêt")
print(train_data[0]['text'][:300])

# Save all three splits to Drive
df_train.to_csv(
    "/content/drive/MyDrive/VisualStep/train_set_1600.csv",
    index=False, encoding='utf-8'
)
df_dev.to_csv(
    "/content/drive/MyDrive/VisualStep/dev_set_200.csv",
    index=False, encoding='utf-8'
)
df_test.to_csv(
    "/content/drive/MyDrive/VisualStep/test_set_200.csv",
    index=False, encoding='utf-8'
)
print(f"Saved train: {len(df_train)} | dev: {len(df_dev)} | test: {len(df_test)}")

# ── CELL 4: Load tokenizer ──

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 4 — Chargement du modèle + LoRA via Unsloth
# ══════════════════════════════════════════════════════════════════════════════
from unsloth import FastModel

# Le bon modèle — version avec Unsloth fixes
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,   # nécessaire pour cette version
)

model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()
print("✅ Modèle rechargé avec Unsloth fixes")

# CELL 5a -  Training config


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
    fp16=True,
    bf16=False,
    report_to="none",
    save_total_limit=1,
    dataloader_num_workers=0,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=dev_data,
    tokenizer=tokenizer,
    args=training_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|user|>\n",
    response_part="<|assistant|>\n",
)
print("✅ Trainer prêt")

# CELL 5b — Training


In [ ]:
import torch
torch._dynamo.config.disable = True

import shutil, os
for item in os.listdir(OUTPUT_DIR):
    item_path = os.path.join(OUTPUT_DIR, item)
    if os.path.isdir(item_path) and 'checkpoint' in item:
        shutil.rmtree(item_path)
        print(f"Supprimé : {item_path}")

print("Démarrage...")
trainer_stats = trainer.train()
print(f"✅ Loss finale : {trainer_stats.metrics['train_loss']:.4f}")

# CELL 6  - Saving LoRa adapters

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 6 — Sauvegarde de l'adaptateur LoRA
# ══════════════════════════════════════════════════════════════════════════════

print("\nSauvegarde de l'adaptateur LoRA...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Sauvegardé dans : {OUTPUT_DIR}")

# (Optionnel) Sauvegarde fusionnée en 16-bit pour inférence autonome
# model.save_pretrained_merged(
#     OUTPUT_DIR + "_merged",
#     tokenizer,
#     save_method="merged_16bit",
# )


# CELL 7a: Loading the model for inference

In [ ]:
# Load saved model for inference
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)

print("Model loaded from:", OUTPUT_DIR)
print("Ready for inference")

# CELL 7b & 8 — Inference on Test Set and Metrics

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 7b — Inférence sur le jeu de test
# ══════════════════════════════════════════════════════════════════════════════

print("\nInférence sur le jeu de test...")

FastModel.for_inference(model)
model.eval()

def generate_step_card(instruction, year, subject, max_new_tokens=256):
    """Génère une carte d'étapes pour une consigne donnée."""
    user_content = f"Year: {year} | Subject: {subject} | Instruction: {instruction}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


results = []
ok = fail = 0

for i, (_, row) in enumerate(df_test.iterrows()):
    if i % 10 == 0:
        print(f"  [{i+1}/{len(df_test)}]", end="", flush=True)

    output = generate_step_card(row['input'], row['year'], row['subject'])

    try:
        parsed     = json.loads(output)
        valid_json = True
        n_steps    = parsed.get('n_steps', len(parsed.get('steps', [])))
        support    = parsed.get('support_level', '')
        ok += 1
    except json.JSONDecodeError:
        valid_json = False
        n_steps    = 0
        support    = ''
        fail += 1

    results.append({
        'id':           row.get('id', f'TEST-{i}'),
        'source':       row.get('source', ''),
        'year':         row['year'],
        'subject':      row['subject'],
        'type':         row.get('type', ''),
        'input':        row['input'],
        'gold_output':  str(row.get('output_json', '')),
        'model_output': output,
        'valid_json':   valid_json,
        'n_steps':      n_steps,
        'support_level': support,
    })

print(f"\n\n  Resultats : {ok} JSON valides | {fail} invalides")

results_path = os.path.join(RESULTS_DIR, "test_outputs_visualstep.csv")
pd.DataFrame(results).to_csv(results_path, index=False, encoding='utf-8')
print(f"Resultats sauvegardes : {results_path}")


# ══════════════════════════════════════════════════════════════════════════════
# CELLULE 8 — Métriques de synthèse
# ══════════════════════════════════════════════════════════════════════════════

df_results = pd.DataFrame(results)

print("\n" + "="*50)
print("   VisualStep — Resultats sur le jeu de test")
print("="*50)
print(f"  Exemples total          : {len(df_results)}")
print(f"  JSON valides            : {df_results['valid_json'].sum()} "
      f"({df_results['valid_json'].mean()*100:.1f}%)")
print(f"  Steps dans [1-5]        : "
      f"{((df_results['n_steps']>=1) & (df_results['n_steps']<=5)).sum()}")
print(f"  Moyenne steps/carte     : "
      f"{df_results[df_results['valid_json']]['n_steps'].mean():.2f}")
print(f"  Distribution support    : "
      f"{df_results['support_level'].value_counts().to_dict()}")
print(f"\n  Resultats complets : {results_path}")
print("="*50)

# ── CELL 9:  BLEU and ROUGE metrics ────────────────

In [ ]:
!pip install rouge-score sacrebleu -q

import json
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU

# Load results
df_results = pd.read_csv(
    "/content/drive/MyDrive/VisualStep/results/test_outputs_visualstep.csv"
)

# Extract action text from JSON for comparison
def extract_actions(json_str):
    """Extract concatenated action text from step card JSON."""
    try:
        data = json.loads(json_str)
        steps = data.get("steps", [])
        return " ".join(
            f"{s.get('action','')} {s.get('detail','')}".strip()
            for s in steps
        )
    except:
        return ""

# Build hypothesis and reference lists
hypotheses = []
references  = []

for _, row in df_results.iterrows():
    hyp = extract_actions(str(row["model_output"]))
    ref = extract_actions(str(row["gold_output"]))
    if hyp and ref:
        hypotheses.append(hyp)
        references.append(ref)

print(f"Valid pairs for evaluation: {len(hypotheses)}")

# ── ROUGE ─────────────────────────────────────────────────────────────────────
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)

r1 = r2 = rL = 0
for hyp, ref in zip(hypotheses, references):
    scores = scorer.score(ref, hyp)
    r1 += scores["rouge1"].fmeasure
    r2 += scores["rouge2"].fmeasure
    rL += scores["rougeL"].fmeasure

n = len(hypotheses)
print(f"\n── ROUGE ──────────────────────────")
print(f"ROUGE-1 : {r1/n:.4f}")
print(f"ROUGE-2 : {r2/n:.4f}")
print(f"ROUGE-L : {rL/n:.4f}")

# ── BLEU ──────────────────────────────────────────────────────────────────────
bleu = BLEU(effective_order=True)
result = bleu.corpus_score(hypotheses, [references])
print(f"\n── BLEU ───────────────────────────")
print(f"BLEU-4  : {result.score:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n── Summary ────────────────────────")
print(f"Model        : VisualStep (Phi-3 Mini fine-tuned)")
print(f"Test examples: {n}")
print(f"ROUGE-1      : {r1/n:.4f}")
print(f"ROUGE-2      : {r2/n:.4f}")
print(f"ROUGE-L      : {rL/n:.4f}")
print(f"BLEU-4       : {result.score:.4f}")

# CELL 10 -  pushing model into hugging face

In [ ]:
# pushing model into hugging face
from huggingface_hub import HfApi, login
from google.colab import userdata

# Login with your token
login(token=userdata.get("HF_visualstep_token"))  # add HF_TOKEN to Colab secrets

# Upload the LoRA adapter
api = HfApi()
api.create_repo(
    repo_id="lsadouk1111/VisualStep",
    repo_type="model",
    exist_ok=True,
    private=False  # public so reviewers can access
)

api.upload_folder(
    folder_path="/content/drive/MyDrive/VisualStep/model",
    repo_id="lsadouk1111/VisualStep",
    repo_type="model",
)
print("Model uploaded to: https://huggingface.co/lsadouk1111/VisualStep")

# CELL 11 - Constraint compliance check on the three confirmed baselines




In [ ]:

import json
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

def check_constraints(df, model_name):
    valid = df[df['valid_json'] == True]
    long_actions = 0
    over_5 = 0

    for _, row in valid.iterrows():
        try:
            d = json.loads(row['model_output'])
            steps = d.get('steps', [])
            if len(steps) > 5:
                over_5 += 1
            for s in steps:
                action = s.get('action', '')
                if len(action.split()) > 6:
                    long_actions += 1
        except:
            pass

    n = len(valid)
    print(f"\n── {model_name} ──────────────────────────")
    print(f"  Valid JSON        : {n}/200")
    print(f"  Steps > 5         : {over_5} violations")
    print(f"  Actions > 6 words : {long_actions} violations")

df_vs   = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_visualstep.csv")
df_phi3 = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_phi3_zeroshot.csv")
df_qwen = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_qwen27b_zeroshot.csv")

check_constraints(df_vs,   "VisualStep (fine-tuned)")
check_constraints(df_phi3, "Zero-shot Phi-3 Mini")
check_constraints(df_qwen, "Zero-shot Qwen3.8 27B")

# CELL 12  - Qualitative Comparison — VisualStep vs. Baselines

In [ ]:
import pandas as pd
import json
from google.colab import drive

drive.mount('/content/drive')

df_vs    = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_visualstep.csv")
df_phi3  = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_phi3_zeroshot.csv")
df_qwen  = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_qwen27b_zeroshot.csv")
df_gemma = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_gemma2_2b_zeroshot.csv")

def parse_steps(json_str):
    try:
        d = json.loads(str(json_str))
        steps = d.get("steps", [])
        lines = []
        for s in steps:
            action = s.get("action", "")
            detail = s.get("detail", "")
            line = f"  {s.get('id','')}) {action}"
            if detail:
                line += f" → {detail}"
            lines.append(line)
        return "\n".join(lines) if lines else "[no steps]"
    except:
        return "[invalid JSON]"

# Chosen examples: SYNTH-1096, SYNTH-1639, SYNTH-1479, REAL-0743
indices = [194, 191, 121, 10]
labels  = [
    "SYNTH-1096 | Materials task (maths book + calculator)",
    "SYNTH-1639 | 5-step English annotation (highlighter + page 45)",
    "SYNTH-1479 | Discussion task (whiteboard + discuss) — Phi-3 ACI violation",
    "REAL-0743  | Call-response routine — all models struggle",
]

for idx, label in zip(indices, labels):
    print(f"\n{'='*70}")
    print(f"EXAMPLE: {label}")
    print(f"INPUT: {df_vs.iloc[idx]['input'][:200]}")
    print(f"\nVisualStep (fine-tuned):")
    print(parse_steps(df_vs.iloc[idx]['model_output']))
    print(f"\nZero-shot Phi-3 Mini:")
    print(parse_steps(df_phi3.iloc[idx]['model_output']))
    print(f"\nZero-shot Qwen3.8 27B:")
    print(parse_steps(df_qwen.iloc[idx]['model_output']))
    print(f"\nZero-shot Gemma 2 2B:")
    print(parse_steps(df_gemma.iloc[idx]['model_output']))